# 🏆 Capstone Project: **AgriReady**
### Turn the raw agricultural dataset into machine-learning-ready data

**Student:** ______________________ &nbsp;&nbsp; **Date:** ______________

Read `PROJECT_BRIEF.md` and `RUBRIC.md` first. This notebook is your **guided workspace**: each step explains the goal, gives hints that point back to the lessons, and ends with a self-check. Cells marked `# TODO` are yours to write. Cells with *no* TODO are context.

**Rules of the game**
1. Never modify the raw file. Work on copies.
2. Anything that *learns from data* (imputer, scaler, encoder, selector, caps) lives **inside** the pipeline and is fitted on **training data only**.
3. Document *why*, not just *what*.
4. Use the real Phase 1 dataset if you have it: the simulated `farm_records_raw.csv` is provided so everyone can compare results. (If you use your own, adapt column names and rules.)

**✍️ Your prediction-moment statement (Step 0)**: *write 3–4 sentences: when will the model be used, which information is available then, which is not, and how will you evaluate?*

> _Your answer here…_

In [ ]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)

---
# Step 0 · Frame the problem (10 minutes, but the most important 10 minutes)

Before touching data, write down **what the model will be used for and when**. Everything in your pipeline depends on it.

> **Use-case for this project: a pre-harvest yield forecast.** *Shortly before harvest, the county office wants to forecast each farm's yield (t/ha) so it can plan storage and transport.*
> At that moment the planting date, soil tests, farm attributes, inputs used, and the season's weather and pest pressure are **known**. The actual harvest date, the sale price and the revenue are **not**.

**Your task in this cell:** run it and read the setup. Then write 3–4 sentences stating the prediction moment in *your own words* in the **Prediction-moment statement** cell at the top of the notebook.

In [ ]:
from pathlib import Path
import re, difflib, joblib
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)       # your deliverables go here
print("Setup complete. Deliverables will be saved in:", OUT.resolve())

---
# Step 1 · Audit the raw data  *(Lessons 0–2)*

**Goal:** an *issues log*: a written, evidence-backed list of everything wrong with the raw file, **before** you fix anything.

**Do this:**
1. Load the raw data and run `profile()`.
2. For each issue family below, write code that *counts* the problem:
   - duplicates (exact and near),
   - wrong types (numbers/dates stored as text),
   - inconsistent category spellings,
   - impossible values (use agronomic common sense),
   - missing values (how much, where, and is it random?),
   - suspicious outliers.
3. Collect the counts in a DataFrame called `issues`.

**Deliverable:** the `issues` table (you will reuse it in the final documentation).

In [ ]:
raw = load_raw_farm()
profile(raw)

# TODO: build a DataFrame called `issues` with one row per problem family and a COUNT as evidence.
# Ideas: raw.duplicated().sum(), raw["crop"].nunique(), regex checks on text columns,
#        rule violations (soil_ph outside 3-10, rainfall <= 0, sentinel codes like -999 / 99 ...)
issues = None

Missing values deserve their own look: *how much, and is it random?*

In [ ]:
# TODO: report the % missing per column, and test at least ONE column for MCAR vs MAR
#       (compare the missing rate between groups of another column).

---
# Step 2 · Clean the data: `clean_farm()`  *(Lessons 1–2)*

**Goal:** a function `clean_farm(raw) -> DataFrame` that performs all **structural** cleaning and learns *nothing* from the data (no means, no medians, no IQR fences; those belong in the pipeline).

**Your function must (in this order):**
1. Convert types: fertilizer → numeric (extract the number), dates → datetime (`format='mixed'`).
2. Standardise `crop`, `county`, `soil_type`, `pest_pressure`; convert `irrigation` to 0/1.
3. Remove duplicates on the natural key `farm_id + planting_date`, keeping the most complete record.
4. Cross-field logic: derive missing `season` from the planting month; blank a harvest date that precedes planting.
5. Invalid values → NaN using a **rule table** (pH 3–10, rainfall 50–3000, temperature −5–40, farm size > 0, fertilizer ≤ 1000, …).
6. Target: drop rows whose yield is missing, ≤ 0, or above the crop's physical ceiling (Maize 12, Beans 4, Wheat 10, Potato 45, Sorghum 9 t/ha).
7. Return a cleaning **log** (how many rows/values each step touched).

**Self-check:** `check_clean(farm_clean)` must pass 13/13.

> 💡 You may re-use ideas and code from Lesson 02, but *understand* every line. Don't copy blindly.

In [ ]:
def clean_farm(raw, verbose=True):
    """Structural cleaning only - must learn NOTHING from the data."""
    log = {"raw rows": len(raw)}
    df = raw.copy()

    # TODO 1. types: fertilizer text -> number; dates -> datetime
    # TODO 2. consistent categories: crop, county, soil_type, pest_pressure (+ irrigation -> 0/1)
    #         hint: normalise (lower, letters only) -> synonyms -> difflib fuzzy match -> NaN
    # TODO 3. remove duplicates on [farm_id, planting_date], keep the most complete row
    # TODO 4. cross-field: derive missing season from planting month; blank harvest < planting
    # TODO 5. rule table: impossible values -> NaN
    # TODO 6. target: drop missing / <= 0 / above crop ceiling
    # TODO 7. record counts in `log` after each step

    if verbose:
        print("CLEANING LOG")
        for k, v in log.items():
            print(f"  {k:50s} {v:>6}")
    return df

Run your function, run the self-check and save the first deliverable.

In [ ]:
farm_clean = clean_farm(load_raw_farm())
check_clean(farm_clean)                    # all 13 checks should pass
farm_clean.to_csv(OUT / "farm_clean.csv", index=False)     # 📦 DELIVERABLE 1: cleaned dataset

---
# Step 3 · Decide what the model may see  *(Lesson 6)*

**Goal:** a dictionary `EXCLUDE` mapping every column you remove to a one-line **reason**, then the objects `X`, `y` (log-yield) and `groups` (farm ids).

Use the *timeline test* from Lesson 6: "would I know this at the prediction moment you wrote in Step 0?" Consider each column. Watch out for: post-harvest columns, columns derived from the target, identifiers, and columns that duplicate other information.

In [ ]:
# TODO: EXCLUDE = {column: "reason", ...}
EXCLUDE = {}
# TODO: build X (predictors), y = log of the target, groups = farm ids
X = y = groups = None

---
# Step 4 · Split first!  *(Lesson 6)*

**Goal:** a hold-out **test set** of unseen **farms** (20%), locked away until Step 10. Use `GroupShuffleSplit`.

**Rule from here on:** nothing is ever `fit` on `X_test`.

In [ ]:
# TODO: grouped 80/20 split -> X_train, X_test, y_train, y_test, groups_train
#       then assert that no farm appears on both sides.

---
# Step 5 · Engineer features  *(Lesson 4)*

**Goal:** at least **8 new features**, covering all of these families, each justified in one sentence:

| Family | Minimum |
|---|---|
| Date-based | 3 (e.g. year, month, sin/cos of month, day of year, rain season) |
| Combination | 3 (ratio, sum, index, interaction, e.g. N:P ratio, NPK total, moisture index) |
| Domain knowledge | 2 (e.g. squared distance from the crop's optimal rainfall/temperature, pH gap) |
| Transform | 1 (e.g. log of farm size) |

**Implementation requirement:** write them as **scikit-learn transformers** (classes with `fit` and `transform`) so they work inside the pipeline. Row-wise formulas learn nothing (`fit` just returns `self`).

You may peek at `utils/agri_prep.py` *after* you've tried: the reference solution uses it.

**Also remember:** time-aware features (rolling/lag) are only valid when built from the **past**. If you attempt the stretch goal (price context from `crop_prices_daily.csv`), use trailing windows.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# TODO: write your own transformers, e.g.
# class MyDateFeatures(BaseEstimator, TransformerMixin): ...
# class MyCombinedFeatures(BaseEstimator, TransformerMixin): ...
# Test each one on X_train.head() before moving on.

---
# Step 6 · Build the preprocessing pipeline  *(Lessons 3 and 7)*

**Goal:** one `Pipeline` object, `pipe`, that takes the **raw-ish** `X` (the output of `clean_farm` minus excluded columns) and ends in a `Ridge` model.

**Required structure:**
```
DateFeatures → AgriFeatures (your feature steps) → ColumnTransformer → SelectKBest → Ridge
```
**ColumnTransformer branches:**
- **numeric:** median imputation → outlier capping learned on train (Winsorizer) → standardisation
- **nominal** (`crop, county, soil_type, rain_season`): most-frequent imputation → one-hot (`handle_unknown='ignore'`)
- **ordinal** (`experience`, `pest_pressure`): most-frequent imputation → ordinal encoding with the **correct explicit order**

**Checkpoints:** the pipeline must `fit` on `X_train` and `predict` on `X_test` without errors, and the transformed training matrix must pass `check_ml_ready`.

In [ ]:
# TODO: define lists of numeric / nominal / ordinal columns (names AFTER your feature steps)
# TODO: def build_pipeline(...): return Pipeline([...])
# TODO: pipe = build_pipeline(); pipe.fit(X_train, y_train)
# TODO: Z_train = pipe[:-1].transform(X_train); check_ml_ready(Z_train)
pipe = None

---
# Step 7 · Select features: with canaries  *(Lesson 5)*

**Goal:** decide how many features to keep, using evidence.

**Do this:**
1. Transform the training data with the preprocessing steps (fitted on train).
2. Append **3 random-noise "canary" columns** and compute mutual information with the target. The best canary score is your **noise floor**: which real features fall below it?
3. Use `GridSearchCV` over `select__k` (e.g. 10, 20, 30, `'all'`) with `GroupKFold(5)` to choose `k`.
4. State your decision in a sentence.

In [ ]:
# TODO 1: add 3 random-noise canary columns to the transformed training matrix, compute mutual information,
#         find the noise floor and list real features below it.
# TODO 2: GridSearchCV over select__k with GroupKFold -> BEST_K
BEST_K = "all"

---
# Step 8 · Evaluate honestly  *(Lessons 6–7)*

**Goal:** show *quantitatively* that your preparation matters.

Compare with **5-fold `GroupKFold` cross-validation on the training set only** (R² on log-yield):

| Model | What it shows |
|---|---|
| Global mean (`DummyRegressor`) | the floor: R² ≈ 0 |
| Crop-only linear model | how much crop alone explains |
| Crude: raw numeric columns → mean-imputer → Ridge | "no preparation" |
| **Your pipeline → Ridge** | the value of Phase 2 |
| Your pipeline → Random Forest | does a stronger model change the story? |

Then run the **shuffled-target sanity test**: shuffle `y_train`, cross-validate your pipeline again. R² must collapse to ≈ 0. If it doesn't, something leaks.

In [ ]:
cv = GroupKFold(5)
# TODO: cross-validate (training data only!) the five models in the table and display them side by side
# TODO: shuffled-target sanity test
results = None

---
# Step 9 · The final, one-shot test  *(Lesson 7)*

Fit the chosen pipeline on the **whole training set**, predict the **held-out farms once**, and report the result in units people understand (t/ha). Resist the urge to go back and tune after seeing this number: the test set has spoken once.

In [ ]:
# TODO: fit on the full training set, predict X_test ONCE, report R² and MAE in t/ha (np.exp of the log predictions)
#       and a per-crop error table.
final = None

---
# Step 10 · ⭐ Stretch goal: price context from the time-series data  *(Lesson 4, Part 3)*

Can *market context at planting time* help? Build `price_trend_30d` = the average daily price of the crop over the **30 days before planting** (Maize, Beans and Potato only; Wheat and Sorghum have no price series → NaN, handled by the imputer).

Rules: **trailing windows only**: nothing from the planting day or later. Test whether the feature improves cross-validated R²; report honestly either way.

In [ ]:
# TODO (optional): build price_trend_30d from crop_prices_daily.csv using ONLY days before planting
#   hints: groupby("crop") + shift(1).rolling(30), then pd.merge_asof(..., by="crop", direction="backward")
#   test it with cross-validation and report whether it helps.

---
# Step 11 · Export the deliverables 📦

| # | Deliverable | File |
|---|---|---|
| 1 | Cleaned dataset | `farm_clean.csv` *(saved in Step 2)* |
| 2 | Preprocessing pipeline (fitted on train) | `preprocessing_pipeline.joblib` (+ full model pipeline `yield_model_pipeline.joblib`) |
| 3 | Engineered features, ML-ready matrices | `X_train_ready.csv`, `X_test_ready.csv`, `y_train.csv`, `y_test.csv`, `feature_dictionary.csv` |
| 4 | Documented explanation of each transformation | `TRANSFORMATION_LOG.md` |

**Important:** the ML-ready matrices are produced by a pipeline **fitted on the training set only**; the test matrix is merely *transformed*.

In [ ]:
# TODO: save with joblib: (a) the fitted preprocessing steps final[:-1]  (b) the full model pipeline
# TODO: X_train_ready = preprocess_only.transform(X_train); X_test_ready = ... (TRANSFORM only!)  -> CSV
# TODO: y_train / y_test -> CSV
# TODO: feature_dictionary.csv  (feature, source column, engineered?, treatment)

### The transformation log
For **every** transformation: *what* it does, *why*, which columns, and whether it **learns from data** (the leakage question). This table is the most valuable part of your documentation: it is what lets a colleague trust and reproduce your work.

In [ ]:
# TODO: write TRANSFORMATION_LOG.md. For EVERY transformation record:
#   step | columns | what it does | why | does it learn from data? (the leakage question)
# Tip: build a DataFrame, then write it as a markdown table into OUT / "TRANSFORMATION_LOG.md".

---
# Step 12 · Final verification ✅
Reload the saved pipeline in a *fresh* state and prove that it works on raw data.

In [ ]:
check_ml_ready(pd.read_csv(OUT / "X_train_ready.csv"))
check_ml_ready(pd.read_csv(OUT / "X_test_ready.csv"))
# TODO: reload your saved pipeline with joblib and predict a messy new record
#       (unseen county, missing pH, missing pest pressure)

---
# 🪞 Reflection (answer in your own words, 2–4 sentences each)

1. Which **single cleaning decision** changed your results the most? How do you know?
2. Name **two features** you excluded for leakage reasons and explain the mechanism in each case.
3. Why did you split **by farm** rather than by random row?
4. Which transformations learn from data? What would have gone wrong if you fitted them on all rows?
5. Where did your pipeline behave **worst** (which crop / which kind of farm)? What would you do next?
6. If the county office wanted a forecast **at planting time** instead of before harvest, what in your pipeline would you change?

---
# ✅ Submission checklist
- [ ] `farm_clean.csv` · `preprocessing_pipeline.joblib` · `yield_model_pipeline.joblib`
- [ ] `X_train_ready.csv`, `X_test_ready.csv`, `y_train.csv`, `y_test.csv`, `feature_dictionary.csv`
- [ ] `TRANSFORMATION_LOG.md` (every transformation: what, why, learns-from-data?)
- [ ] `check_clean` 13/13 and `check_ml_ready` 5/5
- [ ] Shuffled-target test ≈ 0; leakage exclusions documented
- [ ] This notebook runs top-to-bottom without errors